In [1]:
bucket_name ='ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

In [2]:
import boto3
import sagemaker
import sagemaker
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [3]:
import os
import boto3
from sagemaker import get_execution_role

account = boto3.client('sts').get_caller_identity()['Account']
image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu',
#'763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.3.0-cpu-py311-ubuntu20.04-sagemaker'
#'763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.3.0-cpu-py311-ubuntu20.04-sagemaker' #f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu'
role_arn = get_execution_role()

os.environ['I_TEAM_RETAIL'], os.environ['I_CC_RETAIL'] = 'DS RETAIL', '9946100000'
os.environ['I_TEAM_RIESGOS'], os.environ['I_CC_RIESGOS'] = 'DS RIESGOS', '9810200000'

team = 'RETAIL' # TODO 1: Colocar nombre del equipo: RETAIL, RIESGOS
name_ds = 'Hernandez Santiago' # TODO 2: Colocar mis apellidos y nombres
account = boto3.client('sts').get_caller_identity()['Account']

In [4]:
tags = [
    {'Key': 'I_RESPONSABLE_LT', 'Value': name_ds},
    {'Key': 'I_APLICACION', 'Value': 'SDLF'},
    {'Key': 'I_PROYECTO', 'Value': 'SDLF'},
    {'Key': 'I_AMBIENTE', 'Value': 'DEV'},
    {'Key': 'I_CUENTA', 'Value': account},
    {'Key': 'I_SIGLA', 'Value': 'SAN'},
    {'Key': 'I_TEAM', 'Value': os.environ[f'I_TEAM_{team}']},
    {'Key': 'I_CC', 'Value': os.environ[f'I_CC_{team}']},
]

In [5]:
role = sagemaker.get_execution_role()

In [6]:
sklearn_processor = SKLearnProcessor(framework_version='0.20.0',
                                     base_job_name= 'Digitalizacion',
                                     instance_type='ml.r5.12xlarge',
                                     role=role,
                                     tags=tags,
                                     instance_count=1,volume_size_in_gb=30)

In [7]:
import boto3
import sagemaker
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter, IntegerParameter

# Parámetros base
bucket_name = 'ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

# Obtener imagen de XGBoost compatible con SHAP
xgb_image = image_uris.retrieve(framework='xgboost', region=boto3.Session().region_name, version='1.3-1')

# Crear el estimador
xgb = Estimator(
    image_uri=xgb_image,
    role=sagemaker.get_execution_role(),
    instance_count=1,
    instance_type='ml.m5.4xlarge',
    output_path=f's3://{bucket_name}/{model_prefix}/MODEL/output',
    sagemaker_session=sagemaker.Session()
)

# Hiperparámetros base
xgb.set_hyperparameters(
    eval_metric='auc',
    objective='binary:logistic',
    scale_pos_weight=494,
    early_stopping_rounds=200,
    num_round=1000
)

# Espacio de búsqueda
hyperparameter_ranges = {
    'max_depth': IntegerParameter(3, 9),                # igual
    'eta': ContinuousParameter(0.01, 0.25),             # learning_rate
    'subsample': ContinuousParameter(0.6, 1.0),         # igual
    'colsample_bytree': ContinuousParameter(0.6, 1.0),  # igual
    'gamma': ContinuousParameter(0, 7),                 # igual
    'min_child_weight': IntegerParameter(1, 10),        # igual
    'num_round': IntegerParameter(200, 800)             # igual
}

# Crear el tuner
tuner = HyperparameterTuner(
    estimator=xgb,
    objective_metric_name="validation:auc",
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=50,
    max_parallel_jobs=8,
    objective_type="Maximize",
    base_tuning_job_name='hpo-plaft-pj-minorista'
)

# Entradas de datos
s3_input_train = TrainingInput(
    s3_data=f's3://{bucket_name}/{model_prefix}/data_dev_model/train_total.csv',
    content_type='csv'
)

s3_input_val = TrainingInput(
    s3_data=f's3://{bucket_name}/{model_prefix}/data_dev_model/validation_total.csv',
    content_type='csv'
)

# Ejecutar HPO
tuner.fit({'train': s3_input_train, 'validation': s3_input_val}, include_cls_metadata=False)


No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


...........................................................................................................................!


In [9]:
import boto3
import sagemaker
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter, IntegerParameter

In [10]:
!pip install xgboost --prefer-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 68.0 MB/s  0:00:00


In [11]:
!aws s3 cp s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/data_dev_model/validation.csv ./validation_total.csv

download: s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/data_dev_model/validation.csv to ./validation_total.csv


In [12]:
!aws s3 cp s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/MODEL/output/hpo-plaft-pj-minoris-260602-2012-044-9ae0082d/output/model.tar.gz ./model.tar.gz

# Descomprimirlo
#tar -xzf model.tar.gz
# Esto genera un archivo llamado xgboost-model

download: s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/MODEL/output/hpo-plaft-pj-minoris-260602-2012-044-9ae0082d/output/model.tar.gz to ./model.tar.gz


In [13]:
!tar -xzf model.tar.gz

tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'


In [14]:
!pip install xgboost --prefer-binary

In [17]:
import pandas as pd
import xgboost as xgb
import tarfile
import os
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pickle
from scipy.optimize import minimize_scalar  # NUEVO: para optimizar temperature

# ===============================================================
# 1. CARGAR DATOS EXACTAMENTE COMO LO HACES TÚ
# ===============================================================
bucket_name = 'ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'
base_s3_path = f"s3://{bucket_name}/{model_prefix}/data_dev_model"
headers_path = f"{base_s3_path}/headers_total.csv"
val_path = f"{base_s3_path}/validation_total.csv"
extras_val_path = f"{base_s3_path}/extras_validation_total.csv"

# Cargar headers
headers = pd.read_csv(headers_path)
column_order = headers['variables'].tolist()

# Cargar validation sin header, con nombres correctos
df_val = pd.read_csv(val_path, header=None, names=column_order)

# Cargar extras y pegar num_documento, mes_base, tipo_alerta_n2
extras_val = pd.read_csv(extras_val_path)
df_val = pd.concat([df_val.reset_index(drop=True),
                    extras_val[['key_value', 'cod_mes']]], axis=1)

print(f"df_val cargado correctamente. Shape: {df_val.shape}")

# ===============================================================
# 2. PREPROCESAMIENTO (igual que en inferencia)
# ===============================================================
def preprocessing_fn(df: pd.DataFrame) -> pd.DataFrame:
    df = df.fillna(0)
    categorical_columns = [
        "mto_fact_declarado_sunat",
             "cod_ubigeo_cd", "cod_sectorista_id"]
    for col in categorical_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df[categorical_columns] = df[categorical_columns].fillna(df[categorical_columns].mean())
    return df

df_val = preprocessing_fn(df_val)

# ===============================================================
# 3. SEPARAR TARGET Y FEATURES
# ===============================================================
y_val = df_val['target']
exclude_cols = ['target', 'key_value', 'cod_mes']
X_val = df_val.drop(columns=exclude_cols)

print(f"Target: 'target' - Prevalencia: {y_val.mean():.6f}")
print(f"Features usadas: {X_val.shape[1]} columnas")

# ===============================================================
# 4. CARGAR EL MODELO XGBoost
# ===============================================================
booster = xgb.Booster()
booster.load_model('xgboost-model')

# ===============================================================
# 5. OBTENER LOGITS CRUDOS
# ===============================================================
print("Calculando logits crudos...")
dval = xgb.DMatrix(X_val)
raw_logits = booster.predict(dval, output_margin=True)

# ===============================================================
# 6. CALIBRACIÓN PLATT SCALING (como antes)
# ===============================================================
print("Ajustando Platt Scaling (regresión logística)...")
log_reg = LogisticRegression(C=1e9, solver='lbfgs')
log_reg.fit(raw_logits.reshape(-1, 1), y_val)
a = log_reg.coef_[0][0]
b = log_reg.intercept_[0]
print(f"Parámetros Platt: slope (a) = {a:.6f}, intercept (b) = {b:.6f}")

# Logits calibrados con Platt
calibrated_logits = a * raw_logits + b

# ===============================================================
# 7. TEMPERATURE SCALING (NUEVO: para afilar y tener más scores cerca de 1)
# ===============================================================
print("\nOptimizando Temperature Scaling...")
def negative_log_likelihood(temp):
    if temp <= 0:
        return np.inf
    scaled_logits = calibrated_logits / temp
    probs = 1 / (1 + np.exp(-scaled_logits))
    probs = np.clip(probs, 1e-15, 1 - 1e-15)
    nll = -np.mean(y_val * np.log(probs) + (1 - y_val) * np.log(1 - probs))
    return nll

result = minimize_scalar(negative_log_likelihood, bounds=(0.2, 2.0), method='bounded', tol=1e-5)
optimal_temp = result.x
print(f"Temperature óptima (mínima NLL): {optimal_temp:.4f}")

# Prueba diferentes temperaturas para elegir la que más te guste
print("\n=== EFECTO DE DIFERENTES TEMPERATURES ===")
print("Temp    | Media prob   | Máx prob | % >0.8   | % >0.9   | Brier")
print("-" * 65)
for temp in [0.4, 0.5, 0.6, 0.7, 0.8, 1.0, optimal_temp]:
    scaled_logits = calibrated_logits / temp
    probs = 1 / (1 + np.exp(-scaled_logits))
    print(f"{temp:.3f}    | {probs.mean():.6f}    | {probs.max():.4f}   | {100*(probs>0.8).mean():.3f}%  | {100*(probs>0.9).mean():.3f}%  | {brier_score_loss(y_val, probs):.6f}")

# ===============================================================
# 8. GUARDAR: elige tu temperature final aquí
# ===============================================================
# <<< ELIGE EL VALOR QUE MÁS TE GUSTE DE LA TABLA >>>
final_temperature = 0.6  # CAMBIA ESTE VALOR (ej. 0.5 para más agresivo, 0.8 para más suave)

calibration_data = {
    'slope': a,
    'intercept': b,
    'temperature': final_temperature,   # NUEVO
    'feature_columns': X_val.columns.tolist()
}

with open('calibration_params.pkl', 'wb') as f:
    pickle.dump(calibration_data, f)

with tarfile.open('model_calibrado_sharpened.tar.gz', 'w:gz') as tar:
    tar.add('xgboost-model', arcname='xgboost-model')
    tar.add('calibration_params.pkl', arcname='calibration_params.pkl')

print(f"\n¡Listo! Modelo guardado con temperature = {final_temperature}")
print("Archivo: model_calibrado_sharpened.tar.gz")

df_val cargado correctamente. Shape: (57750, 39)
Target: 'target' - Prevalencia: 0.005108
Features usadas: 36 columnas
Calculando logits crudos...
Ajustando Platt Scaling (regresión logística)...
Parámetros Platt: slope (a) = 0.920893, intercept (b) = -5.226686

Optimizando Temperature Scaling...
Temperature óptima (mínima NLL): 0.9984

=== EFECTO DE DIFERENTES TEMPERATURES ===
Temp    | Media prob   | Máx prob | % >0.8   | % >0.9   | Brier
-----------------------------------------------------------------
0.400    | 0.001570    | 0.9287   | 0.009%  | 0.003%  | 0.004420
0.500    | 0.001885    | 0.8863   | 0.005%  | 0.000%  | 0.004347
0.600    | 0.002262    | 0.8470   | 0.003%  | 0.000%  | 0.004286
0.700    | 0.002728    | 0.8126   | 0.002%  | 0.000%  | 0.004238
0.800    | 0.003325    | 0.7831   | 0.000%  | 0.000%  | 0.004200
1.000    | 0.005115    | 0.7363   | 0.000%  | 0.000%  | 0.004166
0.998    | 0.005097    | 0.7366   | 0.000%  | 0.000%  | 0.004166

¡Listo! Modelo guardado con tempe

/tmp/ipykernel_21421/2592560281.py:101: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  result = minimize_scalar(negative_log_likelihood, bounds=(0.2, 2.0), method='bounded', tol=1e-5)


In [18]:
for umbral in [0.45, 0.40, 0.38, 0.35, 0.32, 0.30]:
    casos = (probs > umbral).sum()
    print(f"Umbral {umbral:.2f} → {casos} casos")

Umbral 0.45 → 81 casos
Umbral 0.40 → 118 casos
Umbral 0.38 → 137 casos
Umbral 0.35 → 173 casos
Umbral 0.32 → 214 casos
Umbral 0.30 → 232 casos


In [19]:
import pandas as pd
import xgboost as xgb
import joblib
import tarfile
import os
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.preprocessing import LabelEncoder

# ===============================================================
# 1. CARGAR DATOS EXACTAMENTE COMO LO HACES TÚ
# ===============================================================

bucket_name = 'ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

base_s3_path = f"s3://{bucket_name}/{model_prefix}/data_dev_model"
headers_path = f"{base_s3_path}/headers_total.csv"
val_path = f"{base_s3_path}/validation_total.csv"
extras_val_path = f"{base_s3_path}/extras_validation_total.csv"

# Cargar headers
headers = pd.read_csv(headers_path)
column_order = headers['variables'].tolist()

# Cargar validation sin header, con nombres correctos
df_val = pd.read_csv(val_path, header=None, names=column_order)

# Cargar extras y pegar num_documento, mes_base, tipo_alerta_n2
extras_val = pd.read_csv(extras_val_path)
df_val = pd.concat([df_val.reset_index(drop=True),
                    extras_val[['key_value', 'cod_mes']]], axis=1)

print(f"df_val cargado correctamente. Shape: {df_val.shape}")
print("Primeras columnas:", df_val.columns.tolist()[:10])
print("Últimas columnas:", df_val.columns.tolist()[-10:])

# ===============================================================
# 2. PREPROCESAMIENTO (igual que en inferencia)
# ===============================================================
def preprocessing_fn(df: pd.DataFrame) -> pd.DataFrame:
    df = df.fillna(0)
    categorical_columns = [
        "mto_fact_declarado_sunat",
            "cod_ubigeo_cd", "cod_sectorista_id"]
    for col in categorical_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df[categorical_columns] = df[categorical_columns].fillna(df[categorical_columns].mean())
    return df

df_val = preprocessing_fn(df_val)

# ===============================================================
# 3. SEPARAR TARGET Y FEATURES
# ===============================================================
# La primera columna es 'target'
y_val = df_val['target']

# Features: todas las columnas EXCEPTO target, num_documento, mes_base, tipo_alerta_n2
exclude_cols = ['target', 'key_value', 'cod_mes']
X_val = df_val.drop(columns=exclude_cols)

print(f"Target: 'target' - Prevalencia: {y_val.mean():.6f}")
print(f"Features usadas para calibración: {X_val.shape[1]} columnas")
print("Ejemplo de columnas features:", X_val.columns.tolist()[:10])

from sklearn.linear_model import LogisticRegression
import numpy as np

# ===============================================================
# 4. CARGAR EL MODELO XGBoost (Booster directamente)
# ===============================================================
booster = xgb.Booster()
booster.load_model('xgboost-model')  # archivo descomprimido

# ===============================================================
# 5. OBTENER SCORES CRUDOS (logits) DEL MODELO ORIGINAL
# ===============================================================
print("Calculando scores crudos (logits) en validation...")
dval = xgb.DMatrix(X_val)
raw_logits = booster.predict(dval, output_margin=True)  # ¡Importante: output_margin=True para logits!

# ===============================================================
# 6. CALIBRACIÓN MANUAL CON REGRESIÓN LOGÍSTICA (Platt Scaling)
# ===============================================================
print("Ajustando regresión logística para calibrar probabilidades...")

# Añadimos intercepto (bias)
log_reg = LogisticRegression(C=1e9, solver='lbfgs')  # C muy alto = casi sin regularización
log_reg.fit(raw_logits.reshape(-1, 1), y_val)

# Coeficientes aprendidos
a = log_reg.coef_[0][0]   # slope
b = log_reg.intercept_[0] # intercept

print(f"Parámetros de calibración: slope (a) = {a:.6f}, intercept (b) = {b:.6f}")

# Función para calibrar cualquier score futuro
def calibrate_scores(logits):
    return 1 / (1 + np.exp(-(a * logits + b)))

# ===============================================================
# 7. VERIFICACIÓN
# ===============================================================
raw_scores_prob = 1 / (1 + np.exp(-raw_logits))  # probabilidades originales (las bajas que tenías)
calibrated_scores = calibrate_scores(raw_logits)

print("\n=== RESULTADOS DE CALIBRACIÓN ===")
print(f"AUC original: {roc_auc_score(y_val, raw_scores_prob):.4f}")
print(f"AUC calibrado: {roc_auc_score(y_val, calibrated_scores):.4f}")
print(f"Brier score original: {brier_score_loss(y_val, raw_scores_prob):.4f}")
print(f"Brier score calibrado: {brier_score_loss(y_val, calibrated_scores):.4f}")
print(f"Media scores original: {raw_scores_prob.mean():.6f}")
print(f"Media scores calibrado: {calibrated_scores.mean():.6f}")
print(f"Prevalencia real: {y_val.mean():.6f}")

# ===============================================================
# 8. GUARDAR: el modelo original + los parámetros a y b
# ===============================================================
import pickle

calibration_data = {
    'slope': a,
    'intercept': b,
    'feature_columns': X_val.columns.tolist()  # para consistencia futura
}

with open('calibration_params.pkl', 'wb') as f:
    pickle.dump(calibration_data, f)

# Empaquetar TODO: modelo XGBoost original + parámetros de calibración
with tarfile.open('model_calibrado.tar.gz', 'w:gz') as tar:
    tar.add('xgboost-model', arcname='xgboost-model')              # modelo original
    tar.add('calibration_params.pkl', arcname='calibration_params.pkl')

print("\n¡Listo! Modelo calibrado guardado como 'model_calibrado.tar.gz'")
print("Contiene:")
print("  - xgboost-model (Booster original)")
print("  - calibration_params.pkl (slope e intercept para calibrar)")

df_val cargado correctamente. Shape: (57750, 39)
Primeras columnas: ['target', 'mto_pas_soles', 'imp_trx_abonosefect_6m', 'imp_trx_cargosefe_6m', 'avg_trx_cargostot_3m', 'cnt_trx_cargostot_3m', 'cnt_trx_abonospromtot_3m', 'rat_trx_abonosefectot_1m', 'rat_trx_abonosefectot_3m', 'rat_trx_abonosefectot_9m']
Últimas columnas: ['ratio_cargos_1m_vs_6m', 'share_cp_egresos', 'share_cp_ingresos', 'ros_por_trx_3m', 'ingresos_vs_facturacion', 'pasivo_vs_ingresos', 'ratio_egresos_exterior', 'ratio_ingresos_exterior', 'key_value', 'cod_mes']
Target: 'target' - Prevalencia: 0.005108
Features usadas para calibración: 36 columnas
Ejemplo de columnas features: ['mto_pas_soles', 'imp_trx_abonosefect_6m', 'imp_trx_cargosefe_6m', 'avg_trx_cargostot_3m', 'cnt_trx_cargostot_3m', 'cnt_trx_abonospromtot_3m', 'rat_trx_abonosefectot_1m', 'rat_trx_abonosefectot_3m', 'rat_trx_abonosefectot_9m', 'rat_mntcrgsefetot_1m']
Calculando scores crudos (logits) en validation...
Ajustando regresión logística para calibrar p

In [20]:
!aws s3 cp model_calibrado_sharpened.tar.gz s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/MODEL/calibrado_v1/model.tar.gz

upload: ./model_calibrado_sharpened.tar.gz to s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/MODEL/calibrado_v1/model.tar.gz
